<a href="https://colab.research.google.com/github/ibzsurvey-crypto/NHNNchat/blob/main/bestRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================
# HIERARCHICAL PDF CHATBOT (COLAB)
# Semantic retrieval + PaddleOCR + VLM
# ==============================================

# -----------------------
# 1. INSTALL REQUIRED PACKAGES
# -----------------------
%pip install -q \
pymupdf \
pymupdf4llm \
openai \
paddlepaddle>=3.3 \
"paddleocr[doc-parser]>=3.4.1" \
sentence-transformers \
faiss-cpu \
nest_asyncio \
langchain-text-splitters

In [2]:


# -----------------------
# 2. ENVIRONMENT FIX
# -----------------------
import os
os.environ["FLAGS_use_mkldnn"] = "1"  # Paddle 3.x optimization

# -----------------------
# 3. IMPORTS
# -----------------------
import fitz
import json
import base64
import nest_asyncio
import pymupdf4llm
from paddleocr import PaddleOCR
from openai import AsyncOpenAI
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

nest_asyncio.apply()


In [3]:

# -----------------------
# 4. CONFIGURATION
# -----------------------
PDF_FOLDER = "/content/pdfs"
IMAGE_FOLDER = "/content/page_images"
INDEX_PATH = "/content/node_index.json"

MODEL_NAME = "all-mpnet-base-v2"   # For embeddings (small, fast)
VLM_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
TOP_K_NODES = 5

os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(IMAGE_FOLDER, exist_ok=True)

# -----------------------
# 5. GROQ CLIENT (VLM)
# -----------------------
from google.colab import userdata

client = AsyncOpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [4]:

# -----------------------
# 6. INITIALIZE OCR (PaddleOCR VL)
# -----------------------
print("Initializing PaddleOCR-VL...")
ocr = PaddleOCR(
    use_textline_orientation=True,
    lang='en',
    enable_mkldnn=True
)
print("OCR ready.")


Initializing PaddleOCR-VL...


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and res

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

...

Creating model: ('PP-OCRv5_server_det', None, None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Using official model (en_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

OCR ready.


In [11]:

# -----------------------
# 7. UTILS
# -----------------------
def run_ocr(image_path):
    try:
        result = ocr.ocr(image_path)
        lines = []
        if result and result[0]:
            for line in result[0]:
                text = line[1][0]
                lines.append(text)
        return "\n".join(lines)
    except Exception as e:
        print("OCR failed:", e)
        return ""

def render_page(pdf_path, page_num, output_dir=IMAGE_FOLDER):
    os.makedirs(output_dir, exist_ok=True)
    page_id = f"{os.path.basename(pdf_path)}_p{page_num}"
    image_path = os.path.join(output_dir, f"{page_id}.jpg")
    if not os.path.exists(image_path):
        doc = fitz.open(pdf_path)
        page = doc.load_page(page_num - 1)
        # Reduce the scaling factor to make the image smaller
        pix = page.get_pixmap(matrix=fitz.Matrix(1.0,1.0))
        pix.save(image_path)
        doc.close()
    return image_path


In [13]:

# -----------------------
# 8. BUILD HIERARCHICAL INDEX
# -----------------------
def build_index():
    node_index = []
    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]

    for pdf_file in pdf_files:
        pdf_path = os.path.join(PDF_FOLDER, pdf_file)
        print(f"\nProcessing PDF: {pdf_file}")

        # PyMuPDF4LLM extracts headings and Markdown
        try:
            md_pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
        except:
            md_pages = []

        doc = fitz.open(pdf_path)
        for i, page in enumerate(doc):
            page_num = i+1
            page_id = f"{pdf_file}_p{page_num}"
            image_path = render_page(pdf_path, page_num)

            # Extract text
            md_text = md_pages[i]["text"] if i < len(md_pages) and isinstance(md_pages[i], dict) else ""
            ocr_text = run_ocr(image_path) if len(md_text.strip()) < 30 else ""
            combined_text = md_text + "\n" + ocr_text

            node_index.append({
                "doc_id": pdf_file.replace(".pdf",""),
                "page_num": page_num,
                "page_id": page_id,
                "image_path": image_path,
                "text": combined_text
            })
        doc.close()

    # Save index
    with open(INDEX_PATH, "w") as f:
        json.dump(node_index, f, indent=2)
    print(f"\nNode index saved: {INDEX_PATH}")
    return node_index




In [14]:

# -----------------------
# 9. LOAD OR BUILD INDEX
# -----------------------
def load_index():
    if os.path.exists(INDEX_PATH):
        print("Loading cached node index...")
        with open(INDEX_PATH,"r") as f:
            return json.load(f)
    return build_index()

nodes = load_index()



Processing PDF: PERSONS UNWELL OUTSIDE QUEEN SQUARE BUILDINGS.pdf
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.


Processing PDF: Ketamine IV poster NHNN.pdf
=== Document parser messages ===
                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.


Processing PDF: PRE-PROCEDURE FASTING 2.pdf
=== Document parser messages ===
                                                                                                                        Using Tesseract for OCR processing.
OCR on page.number=0/1.


Processing PDF: CLONIDINE-3 (003).pdf
=== Document parser messages ===
                                                                                                                                                                                    Using Tesseract for OCR processing.
OCR on page.number=0/1.


Processing PDF: THIOPENTAL IN ACUTE BRAIN INJURY.pdf
=== Document p

In [15]:

# -----------------------
# 10. EMBEDDINGS + FAISS
# -----------------------
print("Creating node embeddings...")
embed_model = SentenceTransformer(MODEL_NAME)
node_texts = [n["text"] for n in nodes]
node_embeddings = embed_model.encode(node_texts, convert_to_numpy=True)

dim = node_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(node_embeddings)
print(f"FAISS index ready with {len(nodes)} nodes.")



Creating node embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index ready with 99 nodes.


In [16]:


# -----------------------
# 11. NODE RETRIEVAL
# -----------------------
def semantic_search(query, top_k=TOP_K_NODES):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, top_k)
    return [nodes[i] for i in I[0]]

# -----------------------
# 12. VLM ANSWER
# -----------------------
async def ask_vlm(query, retrieved_nodes):
    content = [{"type":"text","text":f"""
Answer ONLY using the provided pages.
Question:
{query}

Rules:
- Cite DOC ID and PAGE NUMBER
- Use only visible information
- Be concise
"""}]

    for node in retrieved_nodes:
        content.append({"type":"text","text":f"[SOURCE: {node['doc_id']} PAGE {node['page_num']}]"})
        with open(node["image_path"], "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()
        content.append({"type":"image_url", "image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}})

    response = await client.chat.completions.create(
        model=VLM_MODEL,
        messages=[{"role":"user","content":content}],
        temperature=0,
        max_tokens=1024
    )
    return response.choices[0].message.content


In [17]:
# -----------------------
# 13. CHAT FUNCTION
# -----------------------
async def chat(query):
    retrieved_nodes = semantic_search(query)
    print("\nTop retrieved nodes:")
    for n in retrieved_nodes:
        print(f"  {n['doc_id']} page {n['page_num']}")
    answer = await ask_vlm(query, retrieved_nodes)
    print("\n=== Answer ===\n", answer)
    return answer

# -----------------------
# 14. RUN EXAMPLE
# -----------------------
# Reducing TOP_K_NODES to 1 to avoid 'Request Entity Too Large' error
TOP_K_NODES = 3
query = "tell me about escalate protocol?"
await chat(query)


Top retrieved nodes:
  ESCALATE - POST ENDOVASCULAR PROCEDURES JULY 2024 page 1
  CLINICAL EMERGENCIES AT QUEEN SQUARE JULY 2024 page 1
  ADVERSE CLINICAL SIGNS JULY 2024 page 1
  DELIRIUM ASSESSMENT AND MANAGEMENT page 2
  LABETALOL FOR RAPID CONTROL OF HYPERTENSION page 1

=== Answer ===
 The ESCALATE protocol is described on page 1 of the document titled "ESCALATE - POST ENDOVASCULAR PROCEDURES JULY 2024." 

The ESCALATE protocol is activated if routine post-procedure observations identify possible complications. Endovascular treatment/angiogram with groin puncture or insertion/removal of a vascular access device are escalated if any of the following signs/symptoms are present: 
- Swelling/haematoma/bleeding at puncture site 
- Cardiovascular changes - hypotension, tachycardia, cool peripheries
- Abdominal pain/groin pain 
- Retroperitoneal haematoma may present with only abdominal pain
- Sensation of 'POP' in abdomen 
- Leg/thigh pain or swelling on side of groin puncture
- Agitat

'The ESCALATE protocol is described on page 1 of the document titled "ESCALATE - POST ENDOVASCULAR PROCEDURES JULY 2024." \n\nThe ESCALATE protocol is activated if routine post-procedure observations identify possible complications. Endovascular treatment/angiogram with groin puncture or insertion/removal of a vascular access device are escalated if any of the following signs/symptoms are present: \n- Swelling/haematoma/bleeding at puncture site \n- Cardiovascular changes - hypotension, tachycardia, cool peripheries\n- Abdominal pain/groin pain \n- Retroperitoneal haematoma may present with only abdominal pain\n- Sensation of \'POP\' in abdomen \n- Leg/thigh pain or swelling on side of groin puncture\n- Agitation or sweating \n- Tenderness at puncture site \n\n[DOC ID: ESCALATE - POST ENDOVASCULAR PROCEDURES JULY 2024 PAGE 1]'